In [1]:
%%capture
# Data science
import numpy as np
import pandas as pd

# Data visualization
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
%matplotlib inline
from fasteda import fast_eda

from sklearn.model_selection import train_test_split, KFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, roc_auc_score, f1_score, \
    mean_squared_error, mean_absolute_error, r2_score
from lightgbm import LGBMClassifier, plot_importance, LGBMRegressor

import optuna
from optuna.samplers import TPESampler

import math
import warnings
import requests
from datetime import date
from mkl_random.mklrand import shuffle
warnings.filterwarnings('ignore')

# Initialization
## Import

In [2]:
df = pd.read_csv("unified_mycotoxins_fungi_clean.csv")
df2 = pd.read_csv("target_origin_zen_zan_weather_ready_strict_field_storage.csv")

In [3]:
print(f'The Train dataset has {df.shape[0]} rows and {df.shape[1]} columns')
df.describe()

The Train dataset has 11882 rows and 15 columns


,sample_year,toxin_value_raw,toxin_value_standardized_ug_kg
count,11798.000000,4604.000000,4604.000000
mean,2019.655874,105.534331,885.314699
std,6.913805,3873.135828,4751.704980
min,1970.000000,0.000000,0.000000
25%,2020.000000,0.000000,0.000000
50%,2021.000000,0.097900,15.000000
75%,2022.000000,4.257500,259.925000
max,2022.000000,258000.000000,258000.000000


In [5]:
display('Datas:', df, 'Datas2:', df2)

'Datas:'

,sample_id,source_dataset,crop_type,location_country,location_continent,timestamp,sample_year,toxin_name,toxin_value_raw,toxin_unit,toxin_value_standardized_ug_kg,toxin_detected,fungal_species,association_type,original_metadata
0,SUPP_INC_0,Supplementary Material 1 (Included studies),Cowpeas (Vigna unguiculata),Unknown,Africa,2000s,2009.0,Aflatoxins,NaN,ug/kg,NaN,True,A. flavus,Reported,"{""reference"": ""Houssou et al. (2009)"", ""title""..."
1,SUPP_INC_0,Supplementary Material 1 (Included studies),Cowpeas (Vigna unguiculata),Unknown,Africa,2000s,2009.0,Aflatoxins,NaN,ug/kg,NaN,True,Fusarium spp,Reported,"{""reference"": ""Houssou et al. (2009)"", ""title""..."
2,SUPP_INC_0,Supplementary Material 1 (Included studies),Cowpeas (Vigna unguiculata),Unknown,Africa,2000s,2009.0,Fumonisins,NaN,ug/kg,NaN,True,A. flavus,Reported,"{""reference"": ""Houssou et al. (2009)"", ""title""..."
3,SUPP_INC_0,Supplementary Material 1 (Included studies),Cowpeas (Vigna unguiculata),Unknown,Africa,2000s,2009.0,Fumonisins,NaN,ug/kg,NaN,True,Fusarium spp,Reported,"{""reference"": ""Houssou et al. (2009)"", ""title""..."
4,SUPP_INC_1,Supplementary Material 1 (Included studies),Beans (Phaseolus vulgaris),Unknown,Africa,2010s,2019.0,Aflatoxins,NaN,ug/kg,NaN,True,Aspergillus flavus,Known Producer,"{""reference"": ""Ingenbleek et al. (2019)"", ""tit..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11877,MYCO_2797,mycotoxins_clean.csv (Lithuania Cereal Monitor...,Maize grain (feed),Lithuania,Europe,2022-12-14,2022.0,T-2 Toxin,NaN,mg/kg,NaN,False,Fusarium poae,Known Producer,"{""sample_point"": ""E101A"", ""analytical_method"":..."
11878,MYCO_2797,mycotoxins_clean.csv (Lithuania Cereal Monitor...,Maize grain (feed),Lithuania,Europe,2022-12-14,2022.0,T-2 Toxin,NaN,mg/kg,NaN,False,Fusarium langsethiae,Known Producer,"{""sample_point"": ""E101A"", ""analytical_method"":..."
11879,MYCO_2798,mycotoxins_clean.csv (Lithuania Cereal Monitor...,"Cereal grain / crop (feed, code: A0BTV)",Lithuania,Europe,2022-12-05,2022.0,T-2 Toxin,NaN,mg/kg,NaN,False,Fusarium sporotrichioides,Known Producer,"{""sample_point"": ""Transformateur/Fabricant"", ""..."
11880,MYCO_2798,mycotoxins_clean.csv (Lithuania Cereal Monitor...,"Cereal grain / crop (feed, code: A0BTV)",Lithuania,Europe,2022-12-05,2022.0,T-2 Toxin,NaN,mg/kg,NaN,False,Fusarium poae,Known Producer,"{""sample_point"": ""Transformateur/Fabricant"", ""..."


'Datas2:'

,origin_country_code,origin_country_label_fr,reporting_country_code,reporting_country_name,sample_country_code,sample_country_name,sample_date,sample_year,sample_month,sample_day,crop_group,product_name,sampling_point,param_name,zen_detected,zen_value_ug_kg,lod_ug_kg,loq_ug_kg,result_status,result_type,origin_matches_reporting_country,origin_matches_sample_country,sampling_strategy,program_type,sample_method,source_archive,source_file,resId_A
0,CZ,Tchequie,CZ,Czechia,CZ,Czechia,2019-02-20,2019,2,20,barley,Barley grain (feed),Primary production,Zearalenone,no,NaN,NaN,20.0,not_quantified_below_loq,Non Quantified Value ( less than LOQ),yes,yes,Objective sampling,Official (National) programme,NaN,CZ_latest_15410924_contaminants.zip,OCC_CHEMMON2020_CZ_2019.ZIP/OCC_CHEMMON2020_CZ...,CD4F4D661B734AD9E74B94A279E30233
1,CZ,Tchequie,CZ,Czechia,CZ,Czechia,2019-03-18,2019,3,18,barley,Barley grain (feed),Primary production,Zearalenone,no,NaN,NaN,20.0,not_quantified_below_loq,Non Quantified Value ( less than LOQ),yes,yes,Objective sampling,Official (National) programme,NaN,CZ_latest_15410924_contaminants.zip,OCC_CHEMMON2020_CZ_2019.ZIP/OCC_CHEMMON2020_CZ...,41CB40673B11DFD3300D20756321752A
2,CZ,Tchequie,CZ,Czechia,CZ,Czechia,2019-03-26,2019,3,26,barley,Barley grain (feed),Storage,Zearalenone,no,NaN,NaN,20.0,not_quantified_below_loq,Non Quantified Value ( less than LOQ),yes,yes,Objective sampling,Official (National) programme,NaN,CZ_latest_15410924_contaminants.zip,OCC_CHEMMON2020_CZ_2019.ZIP/OCC_CHEMMON2020_CZ...,356CFC6AF1743F0E38EAACE24AD1EE34
3,CZ,Tchequie,CZ,Czechia,CZ,Czechia,2019-10-23,2019,10,23,barley,Barley grain (feed),Storage,Zearalenone,no,NaN,NaN,20.0,not_quantified_below_loq,Non Quantified Value ( less than LOQ),yes,yes,Objective sampling,Official (National) programme,NaN,CZ_latest_15410924_contaminants.zip,OCC_CHEMMON2020_CZ_2019.ZIP/OCC_CHEMMON2020_CZ...,02DB9AA1D3A5A2F7F7383CCCAF04B6E8
4,CZ,Tchequie,CZ,Czechia,CZ,Czechia,2020-02-12,2020,2,12,barley,Barley grain (feed),Storage,Zearalenone,no,NaN,NaN,20.0,not_quantified_below_loq,Non Quantified Value ( less than LOQ),yes,yes,Objective sampling,Official (National) programme,NaN,CZ_latest_15410924_contaminants.zip,OCC_CHEMMON2021_CZ_2020.ZIP/OCC_CHEMMON2021_CZ...,E0EB9B82C93581D42418FEA7CF28BADE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
670,SK,Slovaquie,SK,Slovakia,SK,Slovakia,2023-09-19,2023,9,19,wheat,Common wheat grain,Primary production,Zearalenone,no,NaN,NaN,1.0,not_quantified_below_loq,Non Quantified Value ( less than LOQ),yes,yes,Selective sampling,Official (National) programme,Official sampling,SK_latest_15411469_contaminants.zip,OCC_CHEMMON2024_SK_2023.ZIP/OCC_CHEMMON2024_SK...,B74BFB4877BB720A50800B3B2A778D77
671,SK,Slovaquie,SK,Slovakia,SK,Slovakia,2023-09-29,2023,9,29,wheat,Common wheat grain,Primary production,Zearalenone,no,NaN,3.0,10.0,not_quantified_below_loq,Non Quantified Value ( less than LOQ),yes,yes,Selective sampling,Official (National) programme,Official sampling,SK_latest_15411469_contaminants.zip,OCC_CHEMMON2024_SK_2023.ZIP/OCC_CHEMMON2024_SK...,2A9C36E2DBFAFB64CC108370298CD965
672,SK,Slovaquie,SK,Slovakia,SK,Slovakia,2023-10-03,2023,10,3,wheat,Common wheat grain,Primary production,Zearalenone,no,NaN,NaN,8.0,not_quantified_below_loq,Non Quantified Value ( less than LOQ),yes,yes,Selective sampling,Official (National) programme,Official sampling,SK_latest_15411469_contaminants.zip,OCC_CHEMMON2024_SK_2023.ZIP/OCC_CHEMMON2024_SK...,310177CA51EE0AC37A0EA8389CAB5564
673,SK,Slovaquie,SK,Slovakia,SK,Slovakia,2023-10-23,2023,10,23,wheat,Common wheat grain,Primary production,Zearalenone,no,NaN,NaN,1.0,not_quantified_below_loq,Non Quantified Value ( less than LOQ),yes,yes,Selective sampling,Official (National) programme,Official sampling,SK_latest_15411469_contaminants.zip,OCC_CHEMMON2024_SK_2023.ZIP/OCC_CHEMMON2024_SK...,72554C183712F1986234C00F4A63BFC4


## Summary

In [6]:
def summary(df):
    summ = pd.DataFrame(df.dtypes, columns=['data type'])
    summ['#missing'] = df.isnull().sum().values
    summ['Duplicate'] = df.duplicated().sum()
    summ['#unique'] = df.nunique().values
    desc = pd.DataFrame(df.describe(include='all').transpose())
    if 'min' in desc.columns:
        summ['min'] = desc['min'].values
    if 'max' in desc.columns:
        summ['max'] = desc['max'].values
    if 'mean' in desc.columns:
        summ['avg'] = desc['mean'].values
    if 'std' in desc.columns:
        summ['std dev'] = desc['std'].values
    if 'top' in desc.columns:
        summ['top value'] = desc['top'].values
    if 'freq' in desc.columns:
        summ['Freq'] = desc['freq'].values

    return summ

In [7]:
summary(df).style.background_gradient()

,data type,#missing,Duplicate,#unique,min,max,avg,std dev,top value,Freq
sample_id,str,0,56,3246,nan,nan,nan,nan,SUPP_INC_24,119
source_dataset,str,0,56,4,nan,nan,nan,nan,mycotoxins_clean.csv (Lithuania Cereal Monitoring),7502
crop_type,str,0,56,175,nan,nan,nan,nan,Corn,3280
location_country,str,0,56,23,nan,nan,nan,nan,Lithuania,7502
location_continent,str,0,56,8,nan,nan,nan,nan,Europe,7724
timestamp,str,0,56,583,nan,nan,nan,nan,2022,1990
sample_year,float64,84,56,43,1970.000000,2022.000000,2019.655874,6.913805,nan,nan
toxin_name,str,0,56,58,nan,nan,nan,nan,Aflatoxins,4217
toxin_value_raw,float64,7278,56,840,0.000000,258000.000000,105.534331,3873.135828,nan,nan
toxin_unit,str,0,56,4,nan,nan,nan,nan,µg/kg,4788


# Diagrams
## Showplot

In [9]:
def showplot(columnname, df):
    plt.rcParams['figure.facecolor'] = 'white'
    plt.rcParams['axes.facecolor'] = 'white'
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    ax = ax.flatten()

    value_counts = df[columnname].value_counts()
    labels = value_counts.index.tolist()

    colors = [to_hex(c) for c in sns.color_palette("Set2", len(labels))]
    color_map = dict(zip(labels, colors))

    # Donut
    ax[0].pie(
        value_counts,
        autopct='%1.1f%%',
        textprops={'size': 9, 'color': 'white', 'fontweight': 'bold'},
        colors=[color_map[label] for label in labels],
        wedgeprops=dict(width=0.35),
        startangle=80,
        pctdistance=0.85
    )
    centre_circle = plt.Circle((0, 0), 0.6, fc='white')
    ax[0].add_artist(centre_circle)

    # Bar chart
    sns.countplot(
        data=df,
        y=columnname,
        hue=columnname,
        order=labels,
        hue_order=labels,
        palette=color_map,
        saturation=1,
        dodge=False,
        ax=ax[1],
        legend=False
    )

    for i, v in enumerate(value_counts):
        ax[1].text(v + 1, i, str(v), color='black', fontsize=10, va='center')

    sns.despine(left=True, bottom=True)
    ax[1].set_ylabel(None)
    ax[1].set_xlabel("")
    ax[1].set_xticks([])
    fig.suptitle(columnname, fontsize=15, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 0.85, 1])
    plt.show()

## Dist

In [10]:
def dist(datasets, labels, drop_columns=None, rows=-1, cols=4):
    colors = ['#05b0a3', '#d68c78', '#c7f2a6', '#21deb2', '#deb221', '#de87cb']
    columns_list = datasets[0].select_dtypes(include=['float64', 'int64']).drop(columns=drop_columns).columns
    rows = math.ceil(len(columns_list) / cols)
    fig, axs = plt.subplots(rows, cols, figsize=(24, 5*rows))
    subtitle = f'Distribution for numerical features: {labels[0]}'
    for i in range(1, len(datasets)):
        subtitle += f' vs {labels[i]}'
    plt.suptitle(subtitle, fontsize=16, fontweight='bold')
    axs = axs.flatten()

    for i, col in enumerate(columns_list):
        for j in range(len(datasets)):
            sns.kdeplot(datasets[j][col], ax=axs[i], fill=True, alpha=0.5, linewidth=0.5, color=colors[j], label=labels[j])
            axis_title = f'{col}: {labels[0]} skewness: {datasets[0][col].skew():.2f}'
            for j in range(1, len(datasets)):
                axis_title += f'\n{labels[j]} skewness: {datasets[j][col].skew():.2f}'
            axs[i].set_title(axis_title)
        axs[i].legend()

        plt.tight_layout()

## Feature distribution

In [11]:
def feature_distribution(feature, val_feature, df):
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))

    ax1 = axes[0]
    df_sort = df.groupby(val_feature)[feature].mean().sort_values(ascending=False).index
    sns.barplot(x=feature, y=val_feature, data=df, palette='light:#4caba4_r', order=df_sort,
                estimator=np.mean, ci=None, errwidth=0, ax=ax1)
    for p in ax1.patches:
        ax1.annotate(f'{p.get_width():.2f}', (p.get_x() + p.get_width() / 2., p.get_y() + p.get_height()),
            ha='center', va='center', xytext=(0, 20), textcoords='offset points', fontsize=10, color='black')
    ax1.set_title(f'Mean {feature} by {val_feature}')
    ax1.set_xlabel(feature)
    ax1.set_ylabel('')
    sns.despine(left=True, bottom=True, ax=ax1)

    # Violin Plot
    ax2 = axes[1]
    sns.violinplot(x=feature, y=val_feature, data=df, palette='light:#4caba4_r', order=df_sort, ax=ax2)
    ax2.set_title(f'Distribution of {feature} by {val_feature}')
    ax2.set_ylabel("")
    plt.yticks([])
    sns.despine(left=True, bottom=True, ax=ax2)
    plt.tight_layout()
    plt.show()

## Check outliers

In [12]:
def check_outliers(df, ncols=3, figsize_per_plot=(8, 3)):
    num_cols = df.select_dtypes(include=['float64', 'int64']).columns
    n = len(num_cols)

    if n == 0:
        print("Aucune colonne numérique.")
        return

    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(
        nrows=nrows,
        ncols=ncols,
        figsize=(figsize_per_plot[0] * ncols, figsize_per_plot[1] * nrows)
    )

    if nrows == 1 and ncols == 1:
        axes = [axes]
    else:
        axes = axes.flatten()

    fig.suptitle('Outliers in the data', fontsize=18, fontweight='bold')

    for i, col in enumerate(num_cols):
        sns.boxplot(data=df, x=col, ax=axes[i])
        axes[i].set_title(col)
        axes[i].set_xlabel(col)
        axes[i].grid(False)

    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    fig.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

## Correlation

In [13]:
def correlation_heatmap(datasets, labels):
    fig, axes = plt.subplots(1, 2, figsize=(20, 10))

    for i in range(len(datasets)):
        corr = datasets[i].corr(method='pearson')
        mask = np.triu(np.ones_like(corr))
        sns.heatmap(corr, annot=True, fmt='.2f', mask=mask, cmap='copper_r', cbar=None, linewidth=2, ax=axes[i])
        axes[i].set_title(f'{labels[i]} Dataset', fontsize=16, fontweight='bold')

    plt.tight_layout()
    plt.show()

## Cross tab

In [ ]:
def cross_tab(row_data, col_data, df):
    ct = pd.crosstab(df[row_data], df[col_data])
    plt.figure(figsize=(10, 5))
    sns.heatmap(ct, annot=True, cmap='Blues', fmt='d', cbar=False)
    plt.title(f'{row_data} and {col_data}')
    plt.xlabel('')
    plt.ylabel('')
    plt.show()

# EDA

In [29]:
eda_df = df.copy()
eda_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7502 entries, 0 to 7501
Data columns (total 81 columns):
 #   Column                                       Non-Null Count  Dtype         
---  ------                                       --------------  -----         
 0   sample_id                                    7502 non-null   str           
 1   source_dataset                               7502 non-null   str           
 2   crop_type                                    7502 non-null   str           
 3   location_country                             7502 non-null   str           
 4   location_continent                           7502 non-null   str           
 5   timestamp                                    7502 non-null   datetime64[us]
 6   sample_year                                  7502 non-null   float64       
 7   toxin_name                                   7502 non-null   str           
 8   toxin_value_raw                              7502 non-null   float64       
 9   toxin_un

## Distribution

## Correlation

# Data reduction
## View

## New computed dataframe

In [58]:
summary(sample_df).style.background_gradient()

,data type,#missing,Duplicate,#unique,min,max,avg,std dev,top value,Freq
sample_id,str,0,0,2799,nan,nan,nan,nan,MYCO_3,4
source_dataset,str,0,0,1,nan,nan,nan,nan,mycotoxins_clean.csv (Lithuania Cereal Monitoring),7502
location_country,str,0,0,1,nan,nan,nan,nan,Lithuania,7502
location_continent,str,0,0,1,nan,nan,nan,nan,Europe,7502
timestamp,datetime64[us],0,0,573,2019-12-30 00:00:00,2022-12-23 00:00:00,2021-07-31 11:17:00.207944,nan,nan,nan
sample_year,float64,0,0,4,2019.000000,2022.000000,2021.067182,0.807753,nan,nan
toxin_name,str,0,0,6,nan,nan,nan,nan,aflatoxins,2934
toxin_value_raw,float64,0,0,195,0.000000,1056.000000,5.783019,41.856134,nan,nan
toxin_unit,str,0,0,2,nan,nan,nan,nan,µg/kg,4788
toxin_value_standardized_ug_kg,float64,0,0,164,0.000000,1056.000000,7.732786,46.776482,nan,nan


In [59]:
summary(sample_df2).style.background_gradient()

,data type,#missing,Duplicate,#unique,min,max,avg,std dev,top value,Freq
origin_country_code,str,0,0,6,nan,nan,nan,nan,DK,433
origin_country_label_fr,str,0,0,6,nan,nan,nan,nan,Danemark,433
reporting_country_code,str,0,0,6,nan,nan,nan,nan,DK,433
reporting_country_name,str,0,0,6,nan,nan,nan,nan,Denmark,433
sample_country_code,str,0,0,6,nan,nan,nan,nan,DK,433
sample_country_name,str,0,0,6,nan,nan,nan,nan,Denmark,433
sample_date,datetime64[us],0,0,394,2019-01-14 00:00:00,2023-11-06 00:00:00,2021-08-14 05:24:16,nan,nan,nan
sample_year,int64,0,0,5,2019.000000,2023.000000,2021.133333,1.458628,nan,nan
sample_month,int64,0,0,12,1.000000,12.000000,6.357037,2.801270,nan,nan
sample_day,int64,0,0,31,1.000000,31.000000,15.752593,9.068130,nan,nan


In [60]:
final_df = sample_df.drop(columns=[
    'sample_id',
    'source_dataset',
    'location_country',
    'location_continent',
    'timestamp',
    'sample_year',
    'toxin_value_raw',
    'toxin_unit',
    'fungal_species',
    'association_type',
    'original_metadata',
    'year',
    'month',
    'day',
    'is_feed'
])

In [61]:
sample_df2

,origin_country_code,origin_country_label_fr,reporting_country_code,reporting_country_name,sample_country_code,sample_country_name,sample_date,sample_year,sample_month,sample_day,crop_group,product_name,sampling_point,param_name,zen_detected,zen_value_ug_kg,lod_ug_kg,loq_ug_kg,result_status,result_type,origin_matches_reporting_country,origin_matches_sample_country,sampling_strategy,program_type,sample_method,source_archive,source_file,resId_A,temperature_2m_mean_lag_1,temperature_2m_mean_lag_2,temperature_2m_mean_lag_3,temperature_2m_mean_lag_7,temperature_2m_mean_lag_14,temperature_2m_mean_lag_30,temperature_2m_mean_lag_60,temperature_2m_mean_mean_last_60obs,temperature_2m_mean_std_last_60obs,temperature_2m_mean_min_last_60obs,temperature_2m_mean_max_last_60obs,temperature_2m_mean_median_last_60obs,temperature_2m_max_lag_1,temperature_2m_max_lag_2,temperature_2m_max_lag_3,temperature_2m_max_lag_7,temperature_2m_max_lag_14,temperature_2m_max_lag_30,temperature_2m_max_lag_60,temperature_2m_max_mean_last_60obs,temperature_2m_max_std_last_60obs,temperature_2m_max_min_last_60obs,temperature_2m_max_max_last_60obs,temperature_2m_max_median_last_60obs,temperature_2m_min_lag_1,temperature_2m_min_lag_2,temperature_2m_min_lag_3,temperature_2m_min_lag_7,temperature_2m_min_lag_14,temperature_2m_min_lag_30,temperature_2m_min_lag_60,temperature_2m_min_mean_last_60obs,temperature_2m_min_std_last_60obs,temperature_2m_min_min_last_60obs,temperature_2m_min_max_last_60obs,temperature_2m_min_median_last_60obs,relative_humidity_2m_mean_lag_1,relative_humidity_2m_mean_lag_2,relative_humidity_2m_mean_lag_3,relative_humidity_2m_mean_lag_7,relative_humidity_2m_mean_lag_14,relative_humidity_2m_mean_lag_30,relative_humidity_2m_mean_lag_60,relative_humidity_2m_mean_mean_last_60obs,relative_humidity_2m_mean_std_last_60obs,relative_humidity_2m_mean_min_last_60obs,relative_humidity_2m_mean_max_last_60obs,relative_humidity_2m_mean_median_last_60obs,precipitation_sum_lag_1,precipitation_sum_lag_2,precipitation_sum_lag_3,precipitation_sum_lag_7,precipitation_sum_lag_14,precipitation_sum_lag_30,precipitation_sum_lag_60,precipitation_sum_mean_last_60obs,precipitation_sum_std_last_60obs,precipitation_sum_min_last_60obs,precipitation_sum_max_last_60obs,precipitation_sum_median_last_60obs,rainy_days_last_obs,precipitation_cumsum_last_obs,precipitation_mean_last_obs,high_humidity_days_last_obs,hot_days_last_obs,cold_days_last_obs,day_of_year,is_feed,toxin_name
0,CZ,Tchequie,CZ,Czechia,CZ,Czechia,2019-02-20,2019,2,20,other,Barley grain (feed),Primary production,Zearalenone,no,0.0,0.0,20.0,not_quantified_below_loq,Non Quantified Value ( less than LOQ),yes,yes,Objective sampling,Official (National) programme,NaN,CZ_latest_15410924_contaminants.zip,OCC_CHEMMON2020_CZ_2019.ZIP/OCC_CHEMMON2020_CZ...,CD4F4D661B734AD9E74B94A279E30233,3.11,2.21,2.06,1.76,-5.60,-5.69,4.78,-0.673000,3.081288,-7.60,4.78,0.125,10.82,10.73,9.87,4.81,-0.21,-0.61,7.12,2.572500,3.339529,-3.22,10.82,1.880,-1.86,-2.00,-2.85,-1.32,-9.07,-9.12,3.08,-3.406167,3.657951,-11.37,3.08,-2.210,87.80,81.54,85.13,89.72,95.99,89.72,97.44,91.419833,4.359728,76.43,97.44,92.560,0.17,0.00,0.01,0.35,0.12,0.00,9.28,2.029333,3.203704,0.0,17.86,0.685,55,121.76,2.029333,59,0,60,51,False,zearalenone
1,CZ,Tchequie,CZ,Czechia,CZ,Czechia,2019-03-18,2019,3,18,other,Barley grain (feed),Primary production,Zearalenone,no,0.0,0.0,20.0,not_quantified_below_loq,Non Quantified Value ( less than LOQ),yes,yes,Objective sampling,Official (National) programme,NaN,CZ_latest_15410924_contaminants.zip,OCC_CHEMMON2020_CZ_2019.ZIP/OCC_CHEMMON2020_CZ...,41CB40673B11DFD3300D20756321752A,9.15,6.72,6.18,2.21,7.80,2.31,1.76,1.032833,4.279331,-7.60,9.15,1.915,17.24,9.15,8.88,3.89,15.87,10.49,4.56,5.670667,5.214805,-3.22,17.24,4.920,2.94,4.12,2.26,0.27,3.84,-1.73,-0.94,-2.508167,4.202752,-11.37,4.12,-1.430,80.15,87.53,84.08,80.63,77.29,80.23,89.54,86.617333,6.646434,66.05,97.32,87.885,2.31,6.44,5.12,3.32,0.38,0.00,0.53,1.485333,3.036917,0.0,17.86,0.2

In [62]:
sample_df2['day_of_year'].value_counts()

day_of_year
234    13
131    10
152     8
235     8
229     7
       ..
64      1
215     1
208     1
256     1
310     1
Name: count, Length: 261, dtype: int64

In [63]:
final_df2 = sample_df2.drop(columns=[
    "origin_country_code",
    "origin_country_label_fr",
    "reporting_country_code",
    "reporting_country_name",
    "sample_country_code",
    "sample_country_name",
    "sample_date",
    "sample_year",
    "sample_month",
    "sample_day",
    "product_name",
    "sampling_point",
    "param_name",
    "lod_ug_kg",
    "loq_ug_kg",
    "result_status",
    "result_type",
    "origin_matches_reporting_country",
    "origin_matches_sample_country",
    "sampling_strategy",
    "program_type",
    "sample_method",
    "source_archive",
    "source_file",
    "resId_A",
    "is_feed",
])
final_df2 = final_df2.rename(columns={
    'zen_detected': 'toxin_detected',
    'zen_value_ug_kg': 'toxin_value_standardized_ug_kg'
})

In [64]:
final_df.columns

Index(['toxin_name', 'toxin_value_standardized_ug_kg', 'toxin_detected',
       'temperature_2m_mean_lag_1', 'temperature_2m_mean_lag_2',
       'temperature_2m_mean_lag_3', 'temperature_2m_mean_lag_7',
       'temperature_2m_mean_lag_14', 'temperature_2m_mean_lag_30',
       'temperature_2m_mean_lag_60', 'temperature_2m_mean_mean_last_60obs',
       'temperature_2m_mean_std_last_60obs',
       'temperature_2m_mean_min_last_60obs',
       'temperature_2m_mean_max_last_60obs',
       'temperature_2m_mean_median_last_60obs', 'temperature_2m_max_lag_1',
       'temperature_2m_max_lag_2', 'temperature_2m_max_lag_3',
       'temperature_2m_max_lag_7', 'temperature_2m_max_lag_14',
       'temperature_2m_max_lag_30', 'temperature_2m_max_lag_60',
       'temperature_2m_max_mean_last_60obs',
       'temperature_2m_max_std_last_60obs',
       'temperature_2m_max_min_last_60obs',
       'temperature_2m_max_max_last_60obs',
       'temperature_2m_max_median_last_60obs', 'temperature_2m_min_lag_1',

In [65]:
final_df2.columns

Index(['crop_group', 'toxin_detected', 'toxin_value_standardized_ug_kg',
       'temperature_2m_mean_lag_1', 'temperature_2m_mean_lag_2',
       'temperature_2m_mean_lag_3', 'temperature_2m_mean_lag_7',
       'temperature_2m_mean_lag_14', 'temperature_2m_mean_lag_30',
       'temperature_2m_mean_lag_60', 'temperature_2m_mean_mean_last_60obs',
       'temperature_2m_mean_std_last_60obs',
       'temperature_2m_mean_min_last_60obs',
       'temperature_2m_mean_max_last_60obs',
       'temperature_2m_mean_median_last_60obs', 'temperature_2m_max_lag_1',
       'temperature_2m_max_lag_2', 'temperature_2m_max_lag_3',
       'temperature_2m_max_lag_7', 'temperature_2m_max_lag_14',
       'temperature_2m_max_lag_30', 'temperature_2m_max_lag_60',
       'temperature_2m_max_mean_last_60obs',
       'temperature_2m_max_std_last_60obs',
       'temperature_2m_max_min_last_60obs',
       'temperature_2m_max_max_last_60obs',
       'temperature_2m_max_median_last_60obs', 'temperature_2m_min_lag_1',

## Result

In [66]:
summary(final_df).style.background_gradient()

,data type,#missing,Duplicate,#unique,min,max,avg,std dev,top value,Freq
toxin_name,str,0,5835,6,nan,nan,nan,nan,aflatoxins,2934
toxin_value_standardized_ug_kg,float64,0,5835,164,0.000000,1056.000000,7.732786,46.776482,nan,nan
toxin_detected,bool,0,5835,2,nan,nan,nan,nan,False,6632
temperature_2m_mean_lag_1,float64,0,5835,524,-10.580000,29.540000,11.919038,7.943282,nan,nan
temperature_2m_mean_lag_2,float64,0,5835,517,-10.580000,29.250000,11.768502,7.895275,nan,nan
temperature_2m_mean_lag_3,float64,0,5835,518,-9.220000,29.250000,11.636038,7.849943,nan,nan
temperature_2m_mean_lag_7,float64,0,5835,521,-8.620000,29.540000,11.851830,7.534256,nan,nan
temperature_2m_mean_lag_14,float64,0,5835,520,-10.580000,29.540000,11.804368,7.514210,nan,nan
temperature_2m_mean_lag_30,float64,0,5835,518,-10.580000,29.250000,10.818939,7.450593,nan,nan
temperature_2m_mean_lag_60,float64,0,5835,509,-10.580000,29.540000,10.429880,8.173386,nan,nan


In [67]:
summary(final_df2).style.background_gradient()

,data type,#missing,Duplicate,#unique,min,max,avg,std dev,top value,Freq
crop_group,str,0,100,2,nan,nan,nan,nan,other,629
toxin_detected,str,0,100,2,nan,nan,nan,nan,no,500
toxin_value_standardized_ug_kg,float64,0,100,205,0.000000,2140.080484,21.961175,105.683677,nan,nan
temperature_2m_mean_lag_1,float64,0,100,426,-5.160000,27.450000,11.336978,6.819051,nan,nan
temperature_2m_mean_lag_2,float64,0,100,430,-3.230000,28.190000,11.412963,6.838695,nan,nan
temperature_2m_mean_lag_3,float64,0,100,429,-5.310000,28.950000,11.450874,6.876276,nan,nan
temperature_2m_mean_lag_7,float64,0,100,426,-5.270000,26.980000,11.538207,6.782954,nan,nan
temperature_2m_mean_lag_14,float64,0,100,427,-8.250000,28.630000,11.099896,6.855360,nan,nan
temperature_2m_mean_lag_30,float64,0,100,435,-6.970000,27.860000,10.997422,7.392334,nan,nan
temperature_2m_mean_lag_60,float64,0,100,429,-8.270000,29.070000,10.263881,8.317926,nan,nan


# Pre-processing
## Methods

In [68]:
def get_variable_types(dataframe, exclude=None):
    if exclude is None:
        exclude = []
    continuous_vars = []
    categorical_vars = []

    for column in dataframe.columns:
	    if column not in exclude:
	        categorical_vars.append(column) if dataframe[column].dtype == 'str' else continuous_vars.append(column)

    return continuous_vars, categorical_vars

In [69]:
def confusion():
	plt.figure(figsize=(15, 6))
	conf_matrix = confusion_matrix(y_test, y_pred)
	conf_labels = [f'{i}' for i in range(conf_matrix.shape[0])]
	conf_matrix_df = pd.DataFrame(conf_matrix, columns=conf_labels, index=conf_labels)
	plt.imshow(conf_matrix, interpolation='nearest', cmap=plt.cm.Blues)
	plt.xticks(np.arange(conf_matrix.shape[0]), conf_labels, rotation=45)
	plt.yticks(np.arange(conf_matrix.shape[0]), conf_labels)
	plt.xlabel('Predicted Label')
	plt.ylabel('True Label')
	for i in range(conf_matrix.shape[0]):
	    for j in range(conf_matrix.shape[1]):
	        plt.text(j, i, str(conf_matrix[i, j]), ha='center', va='center', color='black')
	plt.grid(False)
	plt.show()

In [70]:
def feature_importances(model):
	feature_importance = model.feature_importances_
	feature_importance_df = pd.DataFrame({'Feature': X.columns, 'Importance': feature_importance})
	feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)
	plt.figure(figsize=(12, 50))
	sns.barplot(x='Importance', y='Feature', data=feature_importance_df)
	plt.title('Feature Importance')
	plt.xlabel('Importance')
	plt.ylabel('')
	sns.despine(left=True, bottom=True)
	plt.show()

## Model 1
### Encoding

In [ ]:
final_dfs = pd.concat([final_df, final_df2]).drop_duplicates()
final_dfs

In [ ]:
final_dfs["toxin_detected"] = final_dfs["toxin_detected"].replace({
    "yes": True,
    "no": False
}).astype(bool)
final_dfs["toxin_detected"]

In [ ]:
continuous_vars, categorical_vars = get_variable_types(final_dfs, ['toxin_detected'])
display(continuous_vars, categorical_vars)

In [ ]:
final_dfs['toxin_value_standardized_ug_kg'] = final_dfs['toxin_value_standardized_ug_kg'].fillna(0)
sample = final_dfs.copy()
sample.info()

In [ ]:
sample = pd.get_dummies(sample, columns=categorical_vars, drop_first=True)
print(f'The encoded dataset has {sample.shape[0]} rows and {sample.shape[1]} columns')

In [ ]:
sample.columns

### Split

In [ ]:
X = sample.copy()
y = X.pop('toxin_detected')
y_2 = X.pop('toxin_value_standardized_ug_kg')
display(X.shape, y.shape)

In [ ]:
X.columns

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
display(f"Train shape: {X_train.shape}", f"Test shape: {X_test.shape}")

In [ ]:
# showplot('toxin_detected', y)
y_train.value_counts()

In [ ]:
y_train.value_counts()

In [ ]:
# sc = StandardScaler()
# X_train = sc.fit_transform(X_train)
# X_test = sc.transform(X_test)

### Hyperparameters

In [ ]:
# def objective(trial, X, y):
#     # Define parameters to be optimized for the LGBMClassifier
#     param = {
#         "objective": "binary",
#         "metric": "binary_logloss",
#         "verbosity": -1,
#         "boosting_type": "gbdt",
#         "random_state": 42,
#         "learning_rate": trial.suggest_float("learning_rate", 0.001, 0.2),
#         "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
#         "lambda_l1": trial.suggest_float("lambda_l1", 0.005, 0.015),
#         "lambda_l2": trial.suggest_float("lambda_l2", 0.02, 0.06),
#         "max_depth": trial.suggest_int("max_depth", 5, 20),
#         "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 0.9),
#         "subsample": trial.suggest_float("subsample", 0.8, 1.0),
#         "min_child_samples": trial.suggest_int("min_child_samples", 5, 50),
#     }
#
#     cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
#     scores = []
#
#     for train_idx, valid_idx in cv.split(X, y):
#         X_tr, X_val = X.iloc[train_idx], X.iloc[valid_idx]
#         y_tr, y_val = y.iloc[train_idx], y.iloc[valid_idx]
#
#         lgbm_classifier = LGBMClassifier(**param)
#         lgbm_classifier.fit(X_tr, y_tr)
#         y_proba = lgbm_classifier.predict_proba(X_val)[:, 1]
#         scores.append(roc_auc_score(y_val, y_proba))
#
#     return np.mean(scores)
#
# # Train Test split
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
#
# # sampler for Optuna optimization using Tree-structured Parzen Estimator sampler
# sampler = optuna.samplers.TPESampler(seed=42)
#
# # Create a study object, run the optimization process and get best parameters
# study = optuna.create_study(direction="maximize", sampler=sampler)
# study.optimize(lambda trial: objective(trial, X, y), n_trials=100)
# best_params = study.best_params
#
# print('='*50)
# print(f"Best params: {best_params}")
# print(f"Best CV AUC: {study.best_value}")

In [ ]:
best_params = {
    "objective": "binary",
    "metric": "binary_logloss",
    "verbosity": -1,
    "boosting_type": "gbdt",
    "random_state": 42,
    'learning_rate': 0.009743961937494949,
    'n_estimators': 213,
    'lambda_l1': 0.013119235781722067,
    'lambda_l2': 0.03749389007648924,
    'max_depth': 18,
    'colsample_bytree': 0.45560483202639046,
    'subsample': 0.8431540916294574,
    'min_child_samples': 8
}

### Build model

In [ ]:
lgbm_classifier = LGBMClassifier(**best_params)
lgbm_classifier.fit(X_train, y_train)
lgbm_classifier.booster_.save_model("toxin_detection_classifier.txt")

y_pred = lgbm_classifier.predict(X_test)
y_pred_proba = lgbm_classifier.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_pred_proba)
f1 = f1_score(y_test, y_pred)

print(f"Test Accuracy: {acc}")
print(f"Test ROC AUC: {auc}")
print(f"Test F1-score: {f1}")

In [ ]:
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

## Confusion matrix

In [ ]:
confusion()

In [ ]:
feature_importances(lgbm_classifier)

## Model 2
### Encoding

In [ ]:
# continuous_vars_2, categorical_vars_2 = get_variable_types(final_df, ['toxin_detected', 'toxin_value_standardized_ug_kg'])
# display(continuous_vars_2, categorical_vars_2)

In [ ]:
# final_df['toxin_value_standardized_ug_kg'] = final_df['toxin_value_standardized_ug_kg'].fillna(0)
# sample = final_df.copy()
# sample['toxin_detected'] == True
# sample.info()

In [ ]:
# sample = pd.get_dummies(sample, columns=categorical_vars_2, drop_first=True)
# print(f'The encoded dataset has {sample.shape[0]} rows and {sample.shape[1]} columns')

### Split

In [ ]:
# X = sample.copy()
# X.columns
# y = X.pop('toxin_value_standardized_ug_kg')
# X.shape
X_2 = X[y]
y_2 = y_2[y]
display(X_2.shape, y_2.shape)

In [ ]:
X_train_2, X_test_2, y_train_2, y_test_2 = train_test_split(X_2, y_2, test_size=0.2, random_state=42)
display(f"Train shape: {X_train_2.shape}", f"Test shape: {X_test_2.shape}")

In [ ]:
y_train_2.value_counts()

### Hyperparameters

In [ ]:
# def objective(trial, X, y):
#     # Define parameters to be optimized for the LGBMClassifier
#     param = {
#         "objective": "regression",
#         "metric": "rmse",
#         "verbosity": -1,
#         "boosting_type": "gbdt",
#         "random_state": 42,
#         "learning_rate": trial.suggest_float("learning_rate", 0.001, 0.2),
#         "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
#         "lambda_l1": trial.suggest_float("lambda_l1", 0.005, 0.015),
#         "lambda_l2": trial.suggest_float("lambda_l2", 0.02, 0.06),
#         "max_depth": trial.suggest_int("max_depth", 5, 20),
#         "num_leaves": trial.suggest_int("num_leaves", 7, 255),
#         "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 0.9),
#         "subsample": trial.suggest_float("subsample", 0.8, 1.0),
#         "min_child_samples": trial.suggest_int("min_child_samples", 5, 50),
#     }
#
#     cv = KFold(n_splits=5, shuffle=True, random_state=42)
#     scores = []
#
#     for train_idx, valid_idx in cv.split(X_train_2):
#         X_tr, X_val = X_train_2.iloc[train_idx], X_train_2.iloc[valid_idx]
#         y_tr, y_val = y_train_2.iloc[train_idx], y_train_2.iloc[valid_idx]
#
#         lgbm_regressor = LGBMRegressor(**param)
#         lgbm_regressor.fit(X_tr, y_tr)
#         pred = lgbm_regressor.predict(X_val)
#
#         scores.append(mean_squared_error(y_val, pred))
#
#     return np.mean(scores)
#
# # Train Test split
# # X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
#
# # sampler for Optuna optimization using Tree-structured Parzen Estimator sampler
# sampler = optuna.samplers.TPESampler(seed=42)
#
# # Create a study object, run the optimization process and get best parameters
# study = optuna.create_study(direction="maximize", sampler=sampler)
# study.optimize(lambda trial: objective(trial, X_2, y_2), n_trials=50)
# best_params = study.best_params
#
# print('='*50)
# print(f"Best params: {best_params}")

In [ ]:
best_params = {
    "objective": "regression",
    "metric": "rmse",
    "verbosity": -1,
    "boosting_type": "gbdt",
    "random_state": 42,
    'learning_rate': 0.0011634354652036005,
    'n_estimators': 717,
    'lambda_l1': 0.013217597158001457,
    'lambda_l2': 0.03417033228523668,
    'max_depth': 15,
    'num_leaves': 54,
    'colsample_bytree': 0.4581407459012575,
    'subsample': 0.84128095567905,
    'min_child_samples': 48
}

### Build model

In [ ]:
lgbm_regressor = LGBMRegressor(**best_params)
lgbm_regressor.fit(X_train_2, y_train_2)
lgbm_regressor.booster_.save_model("toxin_detection_regressor.txt")

y_pred_2 = lgbm_regressor.predict(X_test_2)

rmse = mean_squared_error(y_test_2, y_pred_2)
mae = mean_absolute_error(y_test_2, y_pred_2)
r2 = r2_score(y_test, y_pred)

print(f"Test RMSE: {rmse}")
print(f"Test MAE: {mae}")
print(f"Test R2-score: {r2}")

In [ ]:
for i in range(len(y_test)):
    print(f"Predicted: {y_pred_2[i]} ; True: {np.array(y_test_2)[i]}")

In [ ]:
min(np.array(y_pred_2))

In [ ]:
confusion()

In [ ]:
feature_importances(lgbm_regressor)